# HeritAgent — EXP-0001 kalibrasyon pilotu (Colab, A100 80 GB)

Bu notebook, EXP-0001'den **önce** yapılan kalibrasyon pilotunu çalıştırır (ADR-0003, ADR-0007 E2, ADR-0008).

**Ne yapar:** G0 ajanı, deney havuzlarının hiçbirinde olmayan 30 ALFWorld oyununu oynar. Bölümler şunlar:
- her oyun sıcaklık 0'da bir kez;
- her ailenin ilk oyunu bir kez daha (6 bölüm; sonuçlar birebir aynı mı diye bakılır);
- sahte bir skill bloğuyla tek bir bölüm (kol-B prompt testi).

Ölçümler pilot manifestinde sabitlenmiştir.

**Hiçbir şey ayarlamaz.** Yalnızca ölçer: başarı oranı, hata oranları, bölüm başına model çağrısı, saatte bölüm sayısı, tekrar belirlenimi. Çıkan sayılar bir EXP-0001 sonucu **değildir**.

**Tahmini süre ve maliyet (M: tahmin, ölçüm değil):** kurulum, model indirme ve sunucu açılışı yaklaşık 20–35 dk, pilot yaklaşık 10–40 dk. Toplam 0.5–1.3 saat; A100'de saatte yaklaşık 7.5 CU ile 4–10 CU.

**Nasıl çalıştırılır:**
1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **A100 GPU**. Varsa "Yüksek RAM"i de seçin.
2. Çalışma zamanı > **Tümünü çalıştır**. Drive izni isteyince onaylayın.
3. Bitince rapor Drive'da `heritagent/pilot/pilot-report.json` olarak durur. **Bu dosyayı bana gönderin.**
4. Son hücre GPU'yu kendiliğinden bırakır (`AUTO_RELEASE`).

Oturum yarıda koparsa "Tümünü çalıştır" deyin; pilot Drive'daki kopyasından devam eder. Pilot bittiyse ikinci hücre durur ve hiçbir şey kurulmaz.

In [ ]:
# Ayarlar
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # pilotun kodu; rapor commit'i kaydeder
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # aynı anda oynanan bölüm sayısı (ADR-0008 D3)
MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH_INVARIANT = True     # sunucu açılmazsa False yapın ve bana bildirin (ADR-0008 D4)
AUTO_RELEASE = True        # bitince ya da bir hata olursa GPU'yu bırak (CU harcamasını durdurur)

In [ ]:
# 0) Bir hücre hata verirse "Tümünü çalıştır" orada durur. AUTO_RELEASE açıksa sunucu
# günlüğü Drive'a kopyalanır ve GPU bırakılır: hatadan sonra boşta bekleyen GPU da CU harcar.
import os, shutil, time


def _on_error(shell, etype, value, tb, tb_offset=None):
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    if not AUTO_RELEASE:
        print("AUTO_RELEASE = False: GPU bağlı kalıyor; işiniz bitince bağlantıyı kesin.")
        return
    if os.path.isdir(DRIVE_ROOT) and os.path.exists("/content/vllm-server.log"):
        saved = f"{DRIVE_ROOT}/logs/vllm-server-{time.strftime('%Y%m%d-%H%M%S')}.log"
        os.makedirs(os.path.dirname(saved), exist_ok=True)
        shutil.copy("/content/vllm-server.log", saved)
        print("Sunucu günlüğü Drive'da:", saved)
    if etype.__name__ != "Bitti":
        print("Bir adım başarısız oldu; GPU bırakılıyor (AUTO_RELEASE). Hata metnini bana gönderin.")
    from google.colab import runtime
    runtime.unassign()


get_ipython().set_custom_exc((Exception,), _on_error)

In [ ]:
# 1) GPU: yalnızca tek bir A100 80 GB (ADR-0008 D1). Başka bir GPU'da hiçbir şey kurulmaz.
import re, subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(query.stdout or query.stderr)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()]
assert len(gpus) == 1 and "A100" in gpus[0][0] and int(float(gpus[0][1])) >= 76000, (
    "Bu notebook yalnızca TEK bir A100 80 GB ile çalışır. Çalışma zamanı > Çalışma zamanı "
    "türünü değiştir > A100 GPU seçin ve yeniden bağlanın.")
header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
CUDA_DRIVER = float(re.search(r"CUDA Version:\s*([0-9.]+)", header).group(1))
print("Sürücünün desteklediği en yüksek CUDA:", CUDA_DRIVER)
assert CUDA_DRIVER >= 12.0, "Sürücü CUDA 12'den eski; kurulum yapılamaz. Bana sürücü sürümünü yazın."

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
DONE = f"{DRIVE_ROOT}/pilot/pilot-report.json"


class Bitti(Exception):
    pass


if os.path.exists(DONE):  # iş zaten bitmiş: kurulum ve GPU harcaması yok
    print(open(DONE).read()[:3000])
    raise Bitti("Pilot zaten bitti; rapor: " + DONE + " (Çalışma zamanı > Bağlantıyı kes ve sil)")

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)


def checkout(ref):
    target = f"origin/{ref}" if subprocess.run(
        ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
        capture_output=True).returncode == 0 else ref
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
    return subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                          text=True, check=True).stdout.strip()


COMMIT = checkout(CODE_REF)
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 4) Kurulum (~5-10 dk): her oturumda AYNI sürümler: vLLM 0.29.0 ve torch 2.13.0 (CUDA 12.9
# yapısı), isteği işleyen kütüphaneler (transformers, tokenizers, huggingface-hub, xgrammar,
# llguidance, jinja2), ALFWorld 0.4.2 ve TextWorld 1.7.0. Koşu başladığı sürümleri kaydeder;
# başka sürümlerle devam etmez (ADR-0008 D4).
import os, signal, subprocess, sys


def install():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "vllm==0.29.0+cu129", "torch==2.13.0+cu129", "torchvision==0.28.0+cu129",
                    "torchaudio==2.11.0+cu129", "transformers==5.19.0", "tokenizers==0.23.3",
                    "huggingface-hub==2.2.0", "xgrammar==0.2.8", "llguidance==1.7.6",
                    "jinja2==3.1.6", "-e", f"{REPO}[alfworld]",
                    "--extra-index-url", "https://wheels.vllm.ai/0.29.0/cu129",
                    "--extra-index-url", "https://download.pytorch.org/whl/cu129"], check=True)


install()
check = subprocess.run([sys.executable, "-c", "import vllm, torch; assert torch.cuda.is_available(), "
                        "'CUDA görünmüyor'; print('vllm', vllm.__version__, 'torch', torch.__version__)"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0, "Kurulum denetimi başarısız: vllm/torch açılmadı ya da CUDA görünmüyor."


def cli(*args):
    """Deneyin bir komutunu çalıştırır ve çıktısını canlı yazar. Komut başarısız olursa hata
    verir; böylece "Tümünü çalıştır" sonraki (para harcayan) hücrelere geçmez.
    Durdur düğmesi: komut, süre sınırındaki gibi durur (sürmekte olan bölümler kaydedilir,
    son kopya Drive'a alınır); ikinci basış onu hemen sonlandırır."""
    command = [sys.executable, "-u", "-m", "evaluation.exp0001.cli", *map(str, args)]
    # kendi süreç grubunda: durdurma sinyali ona yalnızca buradan, bir kez ulaşır
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
            raise
    if process.returncode != 0:
        raise RuntimeError(f"`cli {args[0]}` başarısız oldu (çıkış kodu {process.returncode}); "
                           f"sonraki hücreler çalıştırılmadı.")

In [ ]:
# 5) Drive denetimi (saniyeler): yazma, üzerine yazma, geri okuma, küçük dosya süresi.
cli("mirror-check", "--mirror-dir", f"{DRIVE_ROOT}/pilot/mirror")

In [ ]:
# 6) ALFWorld verisi (~2-5 dk): manifestlerin kurulduğu sürüm 0.2.2, sha256 ile doğrulanır.
# (`alfworld-download` başka bir sürüm indirir; onunla hiçbir oyun açılmaz.) Ardından iki
# manifestin 338 oyununun her biri hash'iyle denetlenir; tutmazsa model indirilmeden durur.
import os
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
cli("fetch-data", "--target", "/content/alfworld-data")
cli("verify-data", "--data-root", DATA_ROOT, "--repo", REPO)

In [ ]:
# 7) Aynı A100 kontrolü, deneyin kendi koduyla (fail-closed).
cli("gpu-check")

In [ ]:
# 8) Model: tek bir revizyon sabitlenir ve dosyaları hash'lenir (~5-10 dk).
# Pilot hub'daki güncel commit'i sabitler ve Drive'a yazar; EXP-0001 koşusu aynısını kullanır.
import json, os
revision_file = f"{DRIVE_ROOT}/model-revision.txt"
revision = open(revision_file).read().strip() if os.path.exists(revision_file) else None
cli("prepare-model", "--model", MODEL, *(["--revision", revision] if revision else []),
    "--out", "/content/prepared.json")
prepared = json.load(open("/content/prepared.json"))
open(revision_file, "w").write(prepared["revision"])
print("Model revizyonu:", prepared["revision"], "| ağırlık hash'i:", prepared["weights"]["sha256"])

In [ ]:
# 9) Model sunucusu (~5-15 dk): vllm serve, batch-invariant çekirdekler açık (ADR-0008 D4).
# Hazır olunca "the model server ... serves" yazar; günlük: /content/vllm-server.log
# Sunucu açılmazsa günlüğün sonu hata çıktısında görünür; tamamı Drive'da logs/ altına kopyalanır.
cli("serve", "--prepared", "/content/prepared.json", "--log", "/content/vllm-server.log",
    "--info", "/content/vllm-server.json", *([] if BATCH_INVARIANT else ["--no-batch-invariant"]))

In [ ]:
# 10) Ön kontrol (2 model çağrısı, hiçbir deney kaydına girmez): ReAct biçimli bir yanıt ve
# miner'ın JSON şemasıyla yapılandırılmış bir yanıt. Miner normalde ancak 540 lifetime
# bölümünden sonra ilk kez çalışır; burada hata verirse lifetime'a CU harcamadan durursunuz.
cli("preflight", "--prepared", "/content/prepared.json")

In [ ]:
# 11) Pilot. Kopyası her dakika Drive'a alınır.
cli("pilot", "--prepared", "/content/prepared.json", "--data-root", DATA_ROOT, "--repo", REPO,
    "--server-info", "/content/vllm-server.json", "--workers", WORKERS,
    "--out-dir", "/content/pilot", "--mirror-dir", f"{DRIVE_ROOT}/pilot/mirror")

In [ ]:
# 12) Özet ve EXP-0001 maliyet projeksiyonu (M: pilotun ortalamalarıyla tahmin).
import json, os, shutil
CU_PER_HOUR = 7.5   # Colab'ın A100 için gösterdiği CU/saat değeriyle değiştirin
shutil.copy("/content/pilot/pilot-report.json", f"{DRIVE_ROOT}/pilot/pilot-report.json")
report = json.load(open("/content/pilot/pilot-report.json"))
m = report["measures"]
print("G0 başarı oranı (sıcaklık 0):", m["g0_success_rate_p0"])
print("Bölüm başına model çağrısı:", m["model_calls_per_episode"])
print("Saatte bölüm:", m["throughput_episodes_per_hour"])
print("Tekrar belirlenimi (hepsi aynı mı):", m["temperature_0_repeat_determinism"]["all_identical"])
print("Kol-B prompt testi geçti mi:", m["arm_b_rendering_smoke_test"]["passed"])
print()
print("Yalnızca bölümler; oturum kurulumları ve miner dahil değil:")
print(f"{'aday/seed':>9} {'etkileşim':>22} {'bölüm':>7} {'çağrı':>9} {'saat':>7} {'CU':>7}")
for s in report["projection"]["scenarios"]:
    hours = s["active_hours_at_pilot_throughput"]
    print(f"{s['candidates_per_seed']:>9} {s['interaction']:>22} {s['episodes']:>7} "
          f"{s['model_calls_at_pilot_mean'] or '-':>9} "
          f"{'-' if hours is None else round(hours, 1):>7} "
          f"{'-' if hours is None else round(hours * CU_PER_HOUR):>7}")
print("\nRapor Drive'da:", f"{DRIVE_ROOT}/pilot/pilot-report.json", "— bu dosyayı gönderin.")

In [ ]:
# Son) GPU'yu bırak: bağlı kaldığı her saat ~7.5 CU harcar. İncelemek isterseniz
# AUTO_RELEASE = False yapın. (Bir hücre hata verirse 0. hücre aynısını yapar.)
if AUTO_RELEASE:
    from google.colab import runtime
    runtime.unassign()

**Sonra:** `pilot-report.json` dosyasını gönderin. Ölçümleri ve projeksiyonu inceleyip hesap tavanı kararını size soracağım (takeover notu §5.12). Ardından `EXP-0001-run.ipynb` ile asıl koşu başlar.